# 03 - Report figures

Figures for the report, drawn from the saved scores in `experiments/*/results/`. Nothing is trained here: run all cells after new runs finish and the figures update. Each figure is also saved as a PNG in `report_figures/` for the report.

Every variant was trained twice (seeds 0 and 1). The figures show the mean and, as faint dots, each seed, so the reader sees how much of a difference is real and how much is chance.

In [ ]:
import json
import os

import matplotlib.pyplot as plt
import numpy as np

from lulc import CLASSES

BLUE, ORANGE, INK, MUTED, GRID = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#e6e5e1"   # validation, test, text, axes
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": MUTED, "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True, "axes.grid.axis": "y",
                     "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True, "font.size": 10})
os.makedirs("report_figures", exist_ok=True)
SPLITS = (("val", BLUE, "validation (10 sites)"), ("test", ORANGE, "test (3 sites)"))


def runs(exp, name):
    """Both seeds of a run, as loaded results files: [seed 0, seed 1]."""
    return [json.load(open(f"experiments/{exp}/results/results_{name}{s}.json")) for s in ("", "_seed1")]


def score(r, split, cls=None):
    """One run's score: split 'val' or 'test'; cls None = mIoU, else that class's IoU."""
    if cls is None:
        return r["val_miou"] if split == "val" else r["miou"]
    return r["val_iou" if split == "val" else "iou"][CLASSES.index(cls)]

## Figure 1: what each step added

The path from the first model on the full dataset to the current best: the baseline (`hybrid`, rerun in `exp1_rerun` so that, like every later run, its best epoch is chosen on the current 10 validation sites), plus the yearly median radar as extra input (context, exp2), plus cleaner labels (AGREE: a filled label is kept only where ESA WorldCover agrees, exp3). Left: mean IoU over all classes. Right: water, the class that broke the first prototype.

**Read with care:** context lifts the **test** sites (mIoU 0.311 -> 0.367, water 0.30 -> 0.68) but leaves the **validation** sites unchanged (mIoU 0.376 -> 0.380, water 0.58 -> 0.56). Only cleaner labels (AGREE) improve both. Test water is fragile: the same baseline scores 0.25 or 0.35 depending on the seed. Some choices (context, earlier the radar filters) were made after looking at test scores, so the test sites are no longer an untouched final exam; an independent confirmation needs fresh test sites.

In [ ]:
steps = [("Baseline", runs("exp1_rerun", "hybrid")),
         ("+ context", runs("exp2_architecture", "hybrid_ctx")),
         ("+ AGREE", runs("exp3_agree_rare", "hybrid_ctx_agree"))]
x = np.arange(len(steps))
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, cls, title in zip(axes, (None, "water"), ("mIoU, all 7 classes", "Water IoU")):
    for split, color, label in SPLITS:
        seeds = np.array([[score(r, split, cls) for r in rs] for _, rs in steps])     # (steps, seeds)
        mean = seeds.mean(1)
        ax.scatter(np.repeat(x, seeds.shape[1]), seeds.ravel(), color=color, s=16, alpha=0.4, zorder=3)
        ax.plot(x, mean, color=color, lw=2, marker="o", ms=7, label=label, zorder=4)
        ax.annotate(f"{mean[-1]:.3f}", (x[-1], mean[-1]), xytext=(9, 7 if split == "val" else -7),   # apart if close
                    textcoords="offset points", va="center", color=INK)
    ax.set_xticks(x, [s for s, _ in steps])
    ax.set_xlim(-0.3, len(steps) - 0.45)
    ax.set_title(title, loc="left", color=INK)
axes[0].set_ylabel("IoU (higher is better)")
axes[0].legend(frameon=False, loc="upper left")
m = lambda rs, split: np.mean([score(r, split) for r in rs])
fig.suptitle(f"Context, then cleaner labels: validation mIoU {m(steps[0][1], 'val'):.3f} → {m(steps[-1][1], 'val'):.3f}, "
             f"test mIoU {m(steps[0][1], 'test'):.3f} → {m(steps[-1][1], 'test'):.3f}", x=0.01, ha="left", fontsize=12, color=INK)
fig.text(0.01, -0.03, "Lines: mean of 2 training seeds. Faint dots: each seed. Validation = 10 sites used to choose models; "
         "test = 3 sites, some choices were made after seeing them (see text).", color=MUTED, fontsize=8)
plt.tight_layout()
fig.savefig("report_figures/fig1_progress.png", dpi=200, bbox_inches="tight")
plt.show()

print(f"{'step':12s}{'val mIoU':>10s}{'test mIoU':>11s}{'val water':>11s}{'test water':>12s}")
for name, rs in steps:
    print(f"{name:12s}" + "".join(f"{np.mean([score(r, s, c) for r in rs]):{w}.3f}"
                                  for s, c, w in (("val", None, 10), ("test", None, 11), ("val", "water", 11), ("test", "water", 12))))

## Figure 2: flooded vegetation needs better labels, not more examples

Oversampling drew training windows with flooded vegetation or bare 3x as often, so the model saw about twice as many flooded-vegetation pixels per epoch (2.2% -> 4.6%). AGREE instead removed filled labels that ESA WorldCover contradicts. If the class were short of examples, oversampling should help; if its labels were the problem, AGREE should. All four runs use context.

In [ ]:
cells = [("Context only", runs("exp2_architecture", "hybrid_ctx")),
         ("+ oversampling\n(2x the examples)", runs("exp3_agree_rare", "hybrid_ctx_over3")),
         ("+ AGREE\n(cleaner labels)", runs("exp3_agree_rare", "hybrid_ctx_agree")),
         ("+ both", runs("exp3_agree_rare", "hybrid_ctx_agree_over3"))]
x, w = np.arange(len(cells)), 0.36
fig, ax = plt.subplots(figsize=(9, 4.4))
for i, (split, color, label) in enumerate(SPLITS):
    seeds = np.array([[score(r, split, "flooded_vegetation") for r in rs] for _, rs in cells])
    pos = x + (i - 0.5) * w
    ax.bar(pos, seeds.mean(1), w, color=color, edgecolor="white", linewidth=1.5, label=label)
    ax.scatter(np.repeat(pos, seeds.shape[1]), seeds.ravel(), color=INK, s=12, alpha=0.55, zorder=3)
    if split == "val":                                             # label the decision set only
        for p, m, top in zip(pos, seeds.mean(1), seeds.max(1)):
            ax.annotate(f"{m:.2f}", (p, max(m, top)), xytext=(0, 6), textcoords="offset points", ha="center", va="bottom", color=INK, fontsize=9)
ax.set_xticks(x, [c for c, _ in cells])
ax.set_ylabel("flooded vegetation IoU (higher is better)")
ax.set_ylim(0, None)
ax.legend(frameon=False, loc="upper left")
ax.set_title("Flooded vegetation responds to cleaner labels, not to more examples", loc="left", color=INK, fontsize=12)
fig.text(0.01, -0.04, "Bars: mean of 2 training seeds. Dots: each seed. Numbers: validation means.", color=MUTED, fontsize=8)
plt.tight_layout()
fig.savefig("report_figures/fig2_flooded_vegetation.png", dpi=200, bbox_inches="tight")
plt.show()

for name, rs in cells:
    print(f"{name.splitlines()[0]:16s} val {np.mean([score(r, 'val', 'flooded_vegetation') for r in rs]):.3f}   "
          f"test {np.mean([score(r, 'test', 'flooded_vegetation') for r in rs]):.3f}")

## Helpers for figures 3-8 (run on the server: saved runs, results files and weights live there)

In [ ]:
import re
import urllib.request

import prototype as P

MONTHS = ["Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr", "May"]
NAMES = [c.replace("_", " ") for c in CLASSES]
COLOR = dict(zip(CLASSES, P.COLORS))                          # the class colours of the app and the maps
BEFORE = "#9a9893"


def run_shares(path, smooth=True):
    """A saved prototype run and its monthly % of the square per class (12 x 7): smoothed as the app shows it, or raw."""
    r = P.load(path)
    return r, P.summarise((r.get("smooth") or r["maps"]) if smooth else r["maps"])["shares"]


def short(month):                                             # "2023-01" -> "Jan 23"
    return MONTHS[(int(month[5:]) - 6) % 12] + " " + month[2:4]


def save(fig, name):
    fig.savefig(f"report_figures/{name}.png", dpi=200, bbox_inches="tight")
    plt.show()

## Figure 3: the 120 sites by role, and the band where the first 50 lay

In [ ]:
LAND = "report_figures/ne_110m_land.geojson"                 # Natural Earth land outlines, public domain
if not os.path.exists(LAND):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_110m_land.geojson", LAND)
src = next("".join(c["source"]) for c in json.load(open("01_data_export.ipynb", encoding="utf-8"))["cells"]
           if "".join(c["source"]).startswith("PROJECT ="))
sites = re.findall(r'\("(\w+)",\s*(-?[\d.]+),\s*(-?[\d.]+),\s*"(\w+)"\)', src)
ROLES = [("train", BLUE, "o", "training"), ("val", ORANGE, "s", "validation"), ("final", "#1a9850", "^", "added for the final model"),
         ("holdout", "#7b3294", "D", "holdout, scored once"), ("test", MUTED, "X", "test")]
fig, ax = plt.subplots(figsize=(11, 5.2))
for f in json.load(open(LAND))["features"]:
    g = f["geometry"]
    for poly in g["coordinates"] if g["type"] == "MultiPolygon" else [g["coordinates"]]:
        ax.fill(*np.array(poly[0]).T, color="#e9e8e4", lw=0)
band = ax.axhspan(-25, 27, color=BLUE, alpha=0.07, lw=0, label="27° N to 25° S: where all of the first 50 sites lay")
for role, col, mk, label in ROLES:
    pts = np.array([(float(lo), float(la)) for n, la, lo, r in sites if r == role])
    ax.scatter(*pts.T, s=36, color=col, marker=mk, edgecolor="white", linewidth=0.6, label=f"{label} ({len(pts)})", zorder=3)
ax.set(xlim=(-180, 180), ylim=(-58, 75), aspect="equal", xticks=[], yticks=[])
ax.grid(False)
for s in ax.spines.values():
    s.set_visible(False)
ax.legend(loc="lower left", frameon=False, fontsize=9)
ax.set_title(f"The {len(sites)} sites by role", loc="left", fontsize=12)
save(fig, "fig3_sites_map")

## Figure 4: what each decision was worth (gain against the bar set before the result)

In [ ]:
# (decision, gain in mIoU, bar it had to clear or None for class-level bars, met?) -- numbers from the READMEs:
# exp4-9 validation (12 sites), exp11 val + final sites (74), exp12 the 62 final sites, exp10 the holdout (14 sites)
D = [("exp4: 50 sites instead of 34", 0.059, 0.0, True),
     ("exp5: InSAR coherence input", 0.010, None, False),       # built and bare had to gain +0.03 each
     ("exp6: yearly spread input", 0.002, None, False),         # trees and crops had to gain +0.03 each
     ("exp7: DeepLabV3+ decoder", 0.006, 0.010, False),
     ("exp8: ResNet-50 encoder", 0.011, 0.010, True),           # not confirmed on the holdout
     ("exp8: SSL4EO pretraining", 0.006, 0.020, False),
     ("exp9: CRF smoothing", 0.001, 0.010, False),
     ("exp11: average 2 seeds", 0.006, 0.005, True),
     ("exp11: average 8 views", 0.003, 0.005, False),
     ("exp11: 3-month mean vs despike", 0.003, 0.0, True),
     ("exp12: Lovász loss", 0.001, 0.010, False),
     ("exp10: 62 new sites (holdout)", 0.058, None, True)]
fig, ax = plt.subplots(figsize=(9, 5.2))
y = np.arange(len(D))[::-1]
ax.barh(y, [g for _, g, _, _ in D], height=0.6, color=[BLUE if m else "#c9c7c1" for *_, m in D])
for yi, (_, g, bar, met) in zip(y, D):
    if bar:
        ax.plot([bar, bar], [yi - 0.38, yi + 0.38], color=INK, lw=1.5)
    ax.text(max(g, bar or 0) + 0.0015, yi, f"+{g:.3f}  {'met' if met else 'not met'}", va="center", fontsize=9,
            color=INK if met else MUTED)
ax.set_yticks(y, [d for d, *_ in D])
ax.set_xlim(0, 0.08)
ax.grid(axis="x", color=GRID); ax.grid(axis="y", visible=False)
ax.set_xlabel("gain in mIoU (black tick: the bar written down before the result)")
ax.set_title("Data and averaging paid off; changes to the training recipe did not", loc="left", fontsize=12)
save(fig, "fig4_decisions")

## Figure 5: the three weaknesses outside the tropics, before and after the new sites

In [ ]:
D5 = [("Massachusetts: trees", "42.5000_-72.6000", "trees"), ("Picardy: water", "49.9000_2.6000", "water"),
      ("Nebraska: bare", "40.9000_-98.4000", "bare")]
fig, axs = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)
for ax, (title, f, cls) in zip(axs, D5):
    i = CLASSES.index(cls)
    for folder, col, ls, label in (("prototype_runs_before", BEFORE, "--", "before: trained on 29 tropical sites"),
                                   ("prototype_runs", BLUE, "-", "after: final model, 103 sites")):
        _, s = run_shares(f"{folder}/{f}_2025-06.pkl.gz", smooth=False)
        ax.plot(MONTHS, s[:, i], color=col, ls=ls, lw=2, marker="o", ms=4, label=label)
    ax.set_title(title, loc="left")
    ax.tick_params(axis="x", labelsize=8)
axs[0].set_ylabel("% of the square, June 2025 to May 2026")
axs[0].legend(frameon=False, fontsize=8)
save(fig, "fig5_diagnosis_before_after")

## Figure 6: the holdout (14 sites, scored once), per model and region, and the final model's confusion matrix

In [ ]:
from lulc import LULCDataset
from run_experiments import confusion, load_run

R = "experiments/{}/results/results_{}.json"
H = {"exp4\nfirst app model\n29 sites": [R.format("exp4_50sites", "hybrid_ctx_agree")],
     "ResNet-50\nfrom scratch\n29 sites": [R.format("exp8_ssl4eo", "hybrid_r50_ctx_agree" + s) for s in ("", "_seed1")],
     "ResNet-50\nSSL4EO\n29 sites": [R.format("exp8_ssl4eo", "hybrid_r50_ssl4eo_ctx_agree" + s) for s in ("", "_seed1")],
     "final model\n103 sites": [R.format("exp10_final", "hybrid_r50_ctx_agree_final" + s) for s in ("", "_seed1")]}
K = [("holdout_miou_tropics", "tropics (6 sites)", ORANGE), ("holdout_miou_new_regions", "Europe and the Americas (8 sites)", BLUE),
     ("holdout_miou", "all 14 sites", INK)]
fig, (ax, cx) = plt.subplots(1, 2, figsize=(14, 4.6), gridspec_kw={"width_ratios": [1.2, 1]})
x = np.arange(len(H))
for j, (key, label, col) in enumerate(K):
    v = [np.mean([json.load(open(f))[key] for f in fs]) for fs in H.values()]
    ax.bar(x + (j - 1) * 0.26, v, width=0.24, color=col, label=label)
    for xi, vi in zip(x, v):
        ax.text(xi + (j - 1) * 0.26, vi + 0.005, f"{vi:.2f}", ha="center", fontsize=8, color=INK)
ax.set_xticks(x, list(H), fontsize=9)
ax.set_ylim(0, 0.6); ax.set_ylabel("mIoU (mean of the seeds)")
ax.legend(frameon=False, fontsize=9, loc="upper left")
ax.set_title("Holdout: the new sites lift the new regions, the tropics hold", loc="left", fontsize=12)
ds = LULCDataset("holdout", context=True)
cm, iou = confusion(load_run("experiments/exp10_final/weights/hybrid_r50_ctx_agree_final.pt", ds.channels), ds)
rows = cm / np.maximum(cm.sum(1, keepdims=True), 1)
cx.imshow(rows, cmap="Blues", vmin=0, vmax=1)
for (i, j), v in np.ndenumerate(rows):
    cx.text(j, i, f"{v:.0%}", ha="center", va="center", fontsize=8, color="white" if v > 0.5 else INK)
cx.set_xticks(range(7), NAMES, rotation=35, ha="right", fontsize=9); cx.set_yticks(range(7), NAMES, fontsize=9)
cx.set_xlabel("model says"); cx.set_ylabel("Dynamic World says"); cx.grid(False)
cx.set_title("Final model (seed 0) on the holdout: each row adds up to 100%", loc="left", fontsize=11)
save(fig, "fig6_holdout")

## Figure 7: change across months at four places (final model, smoothed as in the app)

In [ ]:
D7 = [("Bagre reservoir, Burkina Faso", "11.4800_-0.5500"), ("Tonle Sap, Cambodia", "13.1500_104.1000"),
      ("Satkhira, Bangladesh", "22.5500_89.1000"), ("Lake Debo, Mali", "15.3000_-4.1500")]
fig, axs = plt.subplots(1, 4, figsize=(15, 3.3), sharey=True)
for ax, (title, f) in zip(axs, D7):
    r, s = run_shares(f"prototype_runs/{f}_2025-06.pkl.gz")
    for i in np.argsort(-(np.nanmax(s, 0) - np.nanmin(s, 0)))[:2]:   # the two classes that move most
        ax.plot(MONTHS, s[:, i], color=P.COLORS[i], lw=2, marker="o", ms=4, label=NAMES[i])
    ax.legend(frameon=False, fontsize=8, loc="best")
    ax.set_title(title, loc="left", fontsize=10)
    ax.tick_params(axis="x", labelsize=7)
axs[0].set_ylabel("% of the square, June 2025 to May 2026")
save(fig, "fig7_change_demos")
r = P.load("prototype_runs/11.4800_-0.5500_2025-06.pkl.gz")
res = {**r, "maps": r.get("smooth") or r["maps"]}
fig, axs = plt.subplots(1, 6, figsize=(15, 2.9))
for ax, k in zip(axs, (0, 2, 4, 6, 8, 10)):
    ax.imshow(P.month_image(res, k)); ax.set_title(short(res["months"][k]), fontsize=10); ax.axis("off")
fig.legend([plt.Rectangle((0, 0), 1, 1, color=c) for c in P.COLORS], NAMES, loc="lower center", ncol=7, frameon=False, fontsize=9)
fig.suptitle("Bagre reservoir: bare ground vanishes in the rains and returns as the reservoir shrinks", x=0.01, ha="left", fontsize=11)
save(fig, "fig7b_bagre_strip")

## Figure 8: real change events (stress test); the dashed line is the event

In [ ]:
DW_PALISADES = [49.5, 47.0, 49.9, 54.3, 54.2, 50.1, 51.0, 16.4, 13.6, 13.3, 12.8, 12.7]   # Dynamic World trees %, checked 2026-11-07
D8 = [("Kakhovka dam destroyed, 6 June 2023", "47.5250_34.4000_2023-01", "water", 5),
      ("Tulare Lake returns, spring 2023", "36.0500_-119.8000_2022-10", "water", 5),
      ("Manchar Lake overflows, August 2022", "26.4700_67.6300_2022-04", "water", 4),
      ("Palisades fire, January 2025 (missed)", "34.0700_-118.5500_2024-07", "trees", 6)]
fig, axs = plt.subplots(1, 4, figsize=(15, 3.4), sharey=True)
for ax, (title, f, cls, event) in zip(axs, D8):
    r, s = run_shares(f"prototype_runs/{f}.pkl.gz")
    labels = [short(m) for m in r["months"]]
    ax.plot(labels, s[:, CLASSES.index(cls)], color=COLOR[cls], lw=2, marker="o", ms=4, label=f"model: {cls}")
    if cls == "trees":              # the miss: Dynamic World saw the trees go
        ax.plot(labels, DW_PALISADES, color=COLOR[cls], lw=2, ls="--", label="Dynamic World: trees")
        ax.legend(frameon=False, fontsize=8, loc="lower left")
    ax.axvline(event, color=INK, ls="--", lw=1)
    ax.set_title(title, loc="left", fontsize=10)
    ax.tick_params(axis="x", labelsize=7, rotation=45)
axs[0].set_ylabel("% of the square")
save(fig, "fig8_change_events")
r = P.load("prototype_runs/47.5250_34.4000_2023-01.pkl.gz")
res = {**r, "maps": r.get("smooth") or r["maps"]}
fig, axs = plt.subplots(1, 4, figsize=(12, 3.4))
for ax, k in zip(axs, (4, 5, 6, 11)):
    ax.imshow(P.month_image(res, k)); ax.set_title(short(res["months"][k]), fontsize=10); ax.axis("off")
fig.legend([plt.Rectangle((0, 0), 1, 1, color=c) for c in P.COLORS], NAMES, loc="lower center", ncol=7, frameon=False, fontsize=9)
fig.suptitle("Kakhovka reservoir from Sentinel-1 radar alone: full, drained, bare bed, vegetation returning",
             x=0.01, ha="left", fontsize=11)
save(fig, "fig8b_kakhovka_strip")